In [1]:
# Core data manipulation libraries
import pandas as pd
import numpy as np

In [2]:
# Learning to Rank (LTR) framework and machine learning preprocessors
import xgboost as xgb
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
# PyTorch engine activation for neural tracking calculations
import torch
import ast

In [3]:
# Load the master article database containing text body, titles, timestamps, and metadata tags
articles = pd.read_csv('medium_articles.csv')

In [6]:
from feature_engineer import clean_tags

In [7]:
# Standardize tag layouts across the master repository
articles['formatted_tags'] = articles['tags'].apply(clean_tags)

# Create structured text elements (Document Meta-Strings) to provide comprehensive context
# for bi-encoder embedding tracking and indexing operations
articles['combined_content'] = articles.apply(
    lambda row: f"{row['formatted_tags']} | {row['title']} | {str(row['text'])[:300]}", 
    axis=1
)

## Retrieval Stage

In [8]:
from sentence_transformers import SentenceTransformer, util

# Initialize deep transformer embedding tracking architecture from local storage path
model = SentenceTransformer("/mnt/c/Users/raied/Downloads/MediumArticles/MediumArticles/all_mini_lm", local_files_only=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/mnt/c/Users/raied/Downloads/tproj/lib/python3.10/site-packages/torch/cuda/__init__.py:384: UserWarning: Found GPU0 NVIDIA GeForce RTX 5070 Ti Laptop GPU which is of compute capability (CC) 12.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 5.0 which supports hardware CC >=5.0,<6.0 except {5.3}
- 6.0 which supports hardware CC >=6.0,<7.0 except {6.2}
- 7.0 which supports hardware CC >=7.0,<8.0 except {7.2}
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 13.0, 13.2
  _warn_unsupported_code(d, device_cc, code_ccs)
/mnt/c/Users/raied/Downloads/tproj/lib/python3.10/site-packages/torch/cuda/__init__.py:502: UserWarning: 
NVIDIA GeForc

In [9]:
# Pin network processes cleanly onto operational processing spaces (CPU execution layout fallback)
model.to('cpu')

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
)

### Get embeddings for user intent/sentiment.

In [10]:
# Test intent phrase representation mapping user interests
query = "cuisines to try this summer."
query_embedding = model.encode(query, convert_to_tensor=True)

In [123]:
# Map tracking embeddings systematically using text vectors in mini-batches to prevent memory overflows
corpus_embeddings = model.encode(articles['combined_content'].astype(str).tolist(), show_progress_bar=True, batch_size=128, convert_to_tensor=True)

In [145]:
# Persistence management: save embedded vector matrices to prevent retraining needs
np.save('corpus_embeddings.npy', corpus_embeddings.cpu().numpy())

In [12]:
# Fast reloading of cached vector spaces for fast iteration cycles
corpus_embeddings = np.load('corpus_embeddings.npy')

In [15]:
############# Create an Index in Faiss for vector DB #################
import faiss

# 1. Isolate target numerical tensors to standardize processing matrices
embeddings_np = corpus_embeddings.astype('float32') #.cpu()

# 2. Initialize the index (384 is your embedding dimension)
index = faiss.IndexFlatIP(384) # Use IndexFlatIP for Cosine Similarity (if normalized) or Dot Product

# 3. Add vectors to the index layout structures
index.add(embeddings_np)

# 4. Snapshot vector base indexes out to working directory assets
faiss.write_index(index, "my_index1.faiss")

In [16]:
#-------------------------------------------------------------
# Get most similar document via cosine similarity on document embeddings
#-------------------------------------------------------------

# 1. Get the cosine scores between query and LLM embeddings
cos_scores = util.cos_sim(query_embedding, corpus_embeddings)

# 2. Use torch.topk to get the top 100 scores and their corresponding indices
# dim=1 means we are searching across the columns (the documents)
top_k = 100
top_results = torch.topk(cos_scores, k=top_k, dim=1)

# 3. Extract scores and indices
top_scores = top_results.values[0]     # Top 100 scores
embedding_top_indices = top_results.indices[0]

# -------------------------------------------------------------
# Lexical Candidate Filtering via BM25 Algorithmic Search
# -------------------------------------------------------------
from rank_bm25 import BM25Okapi

# 1. Prepare your corpus: List of lists of words (tokenized)
# Ensure your 'text' column is pre-processed into tokens
tokenized_corpus = [doc.split(" ") for doc in articles['combined_content']]

# 2. Initialize BM25 engine with raw training strings
bm25 = BM25Okapi(tokenized_corpus)

# 3. Retrieve for a query
tokenized_query = query.split(" ")
bm25_scores = bm25.get_scores(tokenized_query)

# 4. Get tracking matrix scores
doc_scores = bm25.get_scores(tokenized_query)

### Articles recommended by BM25 method alone

In [17]:
# Sorting documents to review standard lexical matches
top_10_indices = np.argsort(doc_scores)[::-1][:20]

# 2. Print the top 10 docs using the indices
print("Top 10 Recommended Articles:\n")
for i, idx in enumerate(top_10_indices):
    article = articles.iloc[idx]
    print(f"Rank {i+1} (Score: {doc_scores[idx]:.4f})")
    print(f"Title: {article['title']}")
    print(f"URL: {article['url']}")
    print("-" * 30)

Top 10 Recommended Articles:

Rank 1 (Score: 18.8151)
Title: Sports To Try This Summer — Sporforya
URL: https://medium.com/@sporforya/sports-to-try-this-summer-sporforya-a298c868167e
------------------------------
Rank 2 (Score: 15.0546)
Title: Peaceful Puducherry — Give time a Break
URL: https://medium.com/@mumaktravels/peaceful-puducherry-give-time-a-break-65f5646bba68
------------------------------
Rank 3 (Score: 14.9441)
Title: Random Forest Regression. In this blog we’ll try to understand…
URL: https://medium.com/swlh/random-forest-and-its-implementation-71824ced454f
------------------------------
Rank 4 (Score: 14.9327)
Title: Mykonos: our best adresses to relax this summer
URL: https://medium.com/sphere-travel-club/mykonos-our-best-adresses-to-relax-this-summer-62419172ffd1
------------------------------
Rank 5 (Score: 14.6871)
Title: 3 Clothing Trends you need to try this Fall/Winter
URL: https://medium.com/@malik-har03/3-clothing-trends-you-need-to-try-this-fall-winter-ac861cb

### Articles recommended by the Hybrid (Vector Embedding similarity + BM25 approach)

In [19]:
# Print out the Top 10 combined recommendations to inspect fused retrieval alignment
print("--- Top 10 Hybrid Recommendations (RRF) ---")
for i, (doc_idx, rrf_score) in enumerate(final_rrf_rankings[:100]):
    # Fetch original text using the document index
    article_text = articles['title'].iloc[int(doc_idx)]
    # Truncate text just for a clean print statement
    truncated_text = article_text[:75] + "..." if len(article_text) > 75 else article_text
    
    print(f"Rank {i+1} | Doc Index: {doc_idx} | RRF Score: {rrf_score:.5f} | Text: {truncated_text}")

--- Top 10 Hybrid Recommendations (RRF) ---
Rank 1 | Doc Index: 159431 | RRF Score: 0.01639 | Text: Sports To Try This Summer — Sporforya
Rank 2 | Doc Index: 159466 | RRF Score: 0.01639 | Text: 4 Meat-Free Summer Dishes You Need to Make
Rank 3 | Doc Index: 141236 | RRF Score: 0.01613 | Text: Peaceful Puducherry — Give time a Break
Rank 4 | Doc Index: 177678 | RRF Score: 0.01613 | Text: Veg to Table: Courgette
Rank 5 | Doc Index: 180738 | RRF Score: 0.01587 | Text: Random Forest Regression. In this blog we’ll try to understand…
Rank 6 | Doc Index: 159366 | RRF Score: 0.01587 | Text: 6 Summer Salads You Must Try In Delhi NCR
Rank 7 | Doc Index: 159472 | RRF Score: 0.01562 | Text: Mykonos: our best adresses to relax this summer
Rank 8 | Doc Index: 167802 | RRF Score: 0.01562 | Text: August 16: polenta cakes with shallot, leek, zucchini, and tomato (or, the ...
Rank 9 | Doc Index: 82565 | RRF Score: 0.01538 | Text: 3 Clothing Trends you need to try this Fall/Winter
Rank 10 | Doc Index: 159